In [5]:
# References:
# 1. https://github.com/alirezamika/tutorials/tree/master/qtictactoe
# 2. CS152 Session 8 - [4.2] Policy Search with Q-Learning
# 3. https://realpython.com/tic-tac-toe-python/
# 4. https://mathison.ch/de-ch/blog/installing-python-310-on-mac-osx-13/
# 5. https://www.geeksforgeeks.org/finding-optimal-move-in-tic-tac-toe-using-minimax-algorithm-in-game-theory/
# 6. https://levelup.gitconnected.com/mastering-tic-tac-toe-with-minimax-algorithm-3394d65fa88f
# 7. https://thecodingtrain.com/challenges/154-tic-tac-toe-minimax

import numpy as np
import random
import copy
import matplotlib.pyplot as plt

# Define a TicTacToe agent using the minimax algorithm
class TicTacToe_Minimax_Agent:

    # Initialize the agent with a given search depth for the minimax algorithm
    def __init__(self, max_depth=4):
        self.max_depth = max_depth
        
    # Get a list of available actions (empty cells) in the current state
    @staticmethod
    def get_available_actions(state):
        available_actions = []
        for i in range(4):
            for j in range(4):
                if state[i][j] == 0:
                    available_actions.append((i, j))
        return available_actions

    # Check if the given player has won the game in the current state
    @staticmethod
    def check_winner(state, player):
        for row in state:
            if np.all(row == player):
                return True
        for col in state.T:
            if np.all(col == player):
                return True
        if np.all(np.diag(state) == player) or np.all(np.diag(np.fliplr(state)) == player):
            return True
        return False

    # Update the state by performing an action for the given player
    @staticmethod
    def perform_action(state, action, player):
        new_state = copy.deepcopy(state)
        new_state[action[0]][action[1]] = player
        return new_state

    # Select the best action for the current state using the minimax algorithm
    def make_move(self, state):
        best_action = None
        for depth in range(1, self.max_depth + 1):
            _, current_best_action = self.minimax(state, depth, True, -float("inf"), float("inf"))
            if current_best_action is not None:
                best_action = current_best_action

            # Stop searching if a winning move is found
            if self.eval(self.perform_action(state, best_action, 1)) == 10:
                break

        return best_action
    def heuristic_eval(self, state):
        scores = [0, 0]  # [score for player 1, score for player -1]

        for player in [1, -1]:
            for row in state:
                if np.sum(row == player) == 3 and np.sum(row == 0) == 1:
                    scores[player == 1] += 1

            for col in state.T:
                if np.sum(col == player) == 3 and np.sum(col == 0) == 1:
                    scores[player == 1] += 1

            for diag in [np.diag(state), np.diag(np.fliplr(state))]:
                if np.sum(diag == player) == 3 and np.sum(diag == 0) == 1:
                    scores[player == 1] += 1

        return scores[0] - scores[1]

    # Implementation of the minimax algorithm with alpha-beta pruning
    def minimax(self, state, depth, maximizing_player, alpha, beta):
        if depth == 0 or self.check_winner(state, 1) or self.check_winner(state, -1) or not self.get_available_actions(state):
            return self.eval(state), None

        if maximizing_player:
            max_eval = -float("inf")
            best_action = None
            for action in self.get_available_actions(state):
                new_state = self.perform_action(state, action, 1)
                eval_score, _ = self.minimax(new_state, depth - 1, False, alpha, beta)
                if eval_score > max_eval:
                    max_eval = eval_score
                    best_action = action
                alpha = max(alpha, eval_score)
                if beta <= alpha:
                    break
            return max_eval, best_action
        else:
            min_eval = float("inf")
            best_action = None
            for action in self.get_available_actions(state):
                new_state = self.perform_action(state, action, -1)
                eval_score, _ = self.minimax(new_state, depth - 1, True, alpha, beta)
                if eval_score < min_eval:
                    min_eval = eval_score
                    best_action = action
                beta = min(beta, eval_score)
                if beta <= alpha:
                    break
            return min_eval, best_action

    # Evaluation function to score the state for the minimax algorithm
    def eval(self, state):
        if self.check_winner(state, 1):
            return 10
        elif self.check_winner(state, -1):
            return -10
        else:
            return self.heuristic_eval(state)


In [6]:
# Define a random opponent for the TicTacToe game
class RandomOpponent:
    
    # Get a list of available actions (empty cells) in the current state
    @staticmethod
    def get_available_actions(state):
        available_actions = []
        for i in range(4):
            for j in range(4):
                if state[i][j] == 0:
                    available_actions.append((i, j))
        return available_actions

    # Select a random action from the list of available actions
    def make_move(self, state):
        actions = self.get_available_actions(state)
        return random.choice(actions)

# Simulate multiple games between the minimax agent and the random opponent
def simulate_games(num_games, minimax_agent, minimax_depth=4):
    random_opponent = RandomOpponent()

    minimax_wins = 0
    random_wins = 0
    draws = 0

    # Play a specified number of games
    for _ in range(num_games):
        state = np.zeros((4, 4), dtype=int)
        current_player = 1

        # Continue playing until the game is over
        while True:
            if current_player == 1:
                action = minimax_agent.make_move(state)
            else:
                action = random_opponent.make_move(state)

            state = TicTacToe_Minimax_Agent.perform_action(state, action, current_player)

            if TicTacToe_Minimax_Agent.check_winner(state, current_player):
                if current_player == 1:
                    minimax_wins += 1
                else:
                    random_wins += 1
                break

            if not TicTacToe_Minimax_Agent.get_available_actions(state):
                draws += 1
                break

            current_player *= -1

    # Return the results of the simulated games
    return {
        "minimax_wins": minimax_wins,
        "random_wins": random_wins,
        "draws": draws,
    }

# Calculate the minimax agent's win rate
num_games = 100
results = simulate_games(num_games, TicTacToe_Minimax_Agent(), minimax_depth=4)
minimax_win_rate = results["minimax_wins"] / num_games
print(f"Minimax agent win rate over {num_games} games: {minimax_win_rate:.2%}")

# Print the TicTacToe board with human-readable symbols
def print_board(board):
    symbols = {0: '.', 1: 'X', -1: 'O'}
    for row in board:
        print(' '.join([symbols[cell] for cell in row]))
    print()

# Prompt the human player to make a move
def human_move(state):
    while True:
        try:
            move = input("Enter your move (row, col): ")
            row, col = map(int, move.split(','))
            if state[row][col] == 0:
                return (row, col)
            else:
                print("Invalid move. Cell is already filled. Try again.")
        except ValueError:
            print("Invalid input. Please enter row and col as integers separated by a comma.")
        except IndexError:
            print("Invalid input. Please enter row and col values within the range of 0-3.")

# Play a TicTacToe game between a human player and the AI agent
def play_game(ai_agent):
    state = np.zeros((4, 4), dtype=int)
    current_player = 1 # Human player
    human_symbol, ai_symbol = 1, -1

    # Continue playing until the game is over
    while True:
                # Print the current state of the board
        print_board(state)

        # Let the human player or the AI agent make a move
        if current_player == human_symbol:
            action = human_move(state)
        else:
            action = ai_agent.make_move(state)

        # Update the state of the board with the new move
        state = TicTacToe_Minimax_Agent.perform_action(state, action, current_player)

        # Check if there's a winner
        if TicTacToe_Minimax_Agent.check_winner(state, current_player):
            if current_player == human_symbol:
                print("Congratulations! You won!")
            else:
                print("The AI won.")
            print_board(state)
            break

        # Check if the game is a draw (no more available actions)
        if not TicTacToe_Minimax_Agent.get_available_actions(state):
            print("It's a draw!")
            print_board(state)
            break

        # Switch the current player
        current_player *= -1

# Create the AI agent and start the game
ai_agent = TicTacToe_Minimax_Agent()
print("Tic Tac Toe - You are playing as 'X', the AI is playing as 'O'")
play_game(ai_agent)



Minimax agent win rate over 100 games: 79.00%
Tic Tac Toe - You are playing as 'X', the AI is playing as 'O'
. . . .
. . . .
. . . .
. . . .

Enter your move (row, col): 0,0
X . . .
. . . .
. . . .
. . . .

X O . .
. . . .
. . . .
. . . .

Enter your move (row, col): 1,1
X O . .
. X . .
. . . .
. . . .

X O O .
. X . .
. . . .
. . . .

Enter your move (row, col): 2,2
X O O .
. X . .
. . X .
. . . .

X O O .
. X . .
. . X .
. . . O

Enter your move (row, col): 0,3
X O O X
. X . .
. . X .
. . . O

X O O X
O X . .
. . X .
. . . O

Enter your move (row, col): 1,2
X O O X
O X X .
. . X .
. . . O

X O O X
O X X O
. . X .
. . . O

Enter your move (row, col): 2,1
X O O X
O X X O
. X X .
. . . O

X O O X
O X X O
. X X .
O . . O

Enter your move (row, col): 2,0
X O O X
O X X O
X X X .
O . . O

X O O X
O X X O
X X X O
O . . O

Enter your move (row, col): 3,1
X O O X
O X X O
X X X O
O X . O

It's a draw!
X O O X
O X X O
X X X O
O X O O

